## Narrative Position

**Pipeline step:** Preprocessing 선택과 그 fragility (3/6) — **우리 팀 차별점 layer 1**

**This notebook answers:** *"Kiwi exp_F 토큰화가 ESG 단어를 얼마나 보존하고 얼마나 오염시키는가?"*

**Next:** `03_feature_build_v2.ipynb`

**Linked robustness evidence (★ 이 단계의 핵심 instability 증거):**
- `../memberC/01_tokenizer_robustness_kiwi_vs_okt.ipynb` — Kiwi vs Okt 8지표 비교 (tokenizer instability)
- `../memberC/02_preprocessing_eval.ipynb` — exp_B vs E vs F (contamination vs specificity trade-off)
- `../memberC/01_token_inspection.ipynb` — top-N 토큰을 *직접 읽어* 검증 (평가표 명시 요구)

**Evaluation rubric coverage:** 텍스트 처리

**Why this matters for our team's framing:** preprocessing choice는 mechanical 절차가 아니라 *methodological contribution*이다. 어떤 토큰화/불용어/사전 설정에서 결과가 어떻게 흔들리는지가 우리 instability narrative의 layer 1이다. 노트북 본문에 Decision Box 형식으로 *Alternative → Choice → Justification → Limitation*이 기록되어 있다.

---


# 02 · Preprocess Kiwi v2 — 형태소 분석 + 사용자 사전

**Input**  : `data/03_passages/firm_year_documents_v2.parquet`  
           OR 기존 `data/03_passages/esg_passages.csv`  
**Output** : `data/04_preprocessed/tokenized_v_final.parquet`  
           + `data/00_meta/tokenize_log.csv`

---

## 이 단계가 답하는 한 줄

> "ESG 어휘를 정확히 측정하기 위해 어떤 토큰화 전략을 선택했는가?"

## Decision Box — Tokenizer 채택

| 항목 | 내용 |
|---|---|
| **Alternative** | Kiwi / Okt / Mecab-ko / KoBERT subword |
| **Choice** | **Kiwi PRIMARY** |
| **Justification** | 사용자 사전(NNP, score=50)으로 seed 30개 100% 보존 가능. 재현성 우수. |
| **Limitation** | Mecab 대비 미세하게 느림. N=10,000+ 시 재고 필요. |

## 사용자 사전 등록 원칙

복합 명사 (재생에너지, 감사위원회 등)는 Kiwi 기본 분석에서 분리될 수 있다.  
→ `NNP (고유명사)` + `score=50` 으로 사전 등록하면 분리를 막는다.  
→ 결과: seed 30개 corpus 보존율 100% 달성

In [ ]:
import sys, time, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline_config import (
    D_META, D_PASSAGES, D_PREPROC,
    ID_COLS, RANDOM_SEED, SEED_30,
    G_SIGNAL_SET, BOILERPLATE_SET,
    ROBUSTNESS_EXP, PRIMARY_EXP,
)

SEED_SET = set(SEED_30)
print(f"Kiwi tokenization pipeline")
print(f"Seed 30: {SEED_30}")

## 1. Kiwi 초기화 + 사용자 사전 등록

In [ ]:
try:
    from kiwipiepy import Kiwi
    kiwi = Kiwi()
    
    # 사용자 사전 등록 — seed 30개 + 핵심 복합어
    # NNP (고유명사), score=50 → 분리 방지
    USER_DICT_TERMS = SEED_30 + [
        # seed 확장 — 복합어 보강
        "ESG위원회", "ESG경영", "ESG전략",
        "탄소배출권", "탄소발자국", "온실가스감축",
        "안전보건관리", "중대재해처벌법",
        "공급망실사", "공급망리스크",
        "이사회다양성", "사외이사독립성",
        "지속가능경영", "지속가능발전",
    ]
    
    for term in USER_DICT_TERMS:
        kiwi.add_user_word(term, 'NNP', 50)
    
    print(f"Kiwi 초기화 완료")
    print(f"사용자 사전 등록: {len(USER_DICT_TERMS)}개")
    
    # 등록 검증
    test_text = "재생에너지와 감사위원회의 탄소중립 목표"
    test_tokens = [t.form for t in kiwi.tokenize(test_text) if t.tag.startswith('NN')]
    print(f"\n검증: '{test_text}'")
    print(f"  토큰: {test_tokens}")
    
    seed_preserved = [s for s in ['재생에너지','감사위원회','탄소중립'] if s in test_tokens]
    print(f"  seed 보존: {seed_preserved} ({'✓' if len(seed_preserved)==3 else '✗'})")    
    
    KIWI_AVAILABLE = True
except ImportError:
    print("[WARNING] kiwipiepy 없음")
    print("설치: pip install kiwipiepy")
    KIWI_AVAILABLE = False

## 2. 불용어 설정 (preprocessor.py 재사용)

In [ ]:
# preprocessor.py에서 설정 가져오기 (재사용)
try:
    from src.preprocessor import (
        STOPWORDS_EXTENDED, BOILERPLATE_NOUNS, G_SIGNAL_PROTECT,
        filter_boilerplate_sentences, apply_number_mode, EXPERIMENT_CONFIGS,
    )
    print("src.preprocessor 로드 완료")
except ImportError:
    from pipeline_config import G_SIGNAL_SET
    G_SIGNAL_PROTECT = G_SIGNAL_SET
    STOPWORDS_EXTENDED = {
        "기준","기간","항목","결과","과정","단계","수준","규모","범위","목적",
        "주요","전반","향후","기존","신규","상황","조건","운영","관리","수행",
        "비용","금액","자산","이익","투자","연도","분기","당사","회사","기업",
        "및","또한","이를","통해","위해","관련","대한","등","있다","하다",
        "되다","이다","것","수","그","해당","경우","현재","기타",
    } - G_SIGNAL_SET
    BOILERPLATE_NOUNS = BOILERPLATE_SET
    print("pipeline_config fallback 사용")

# G_SIGNAL 보호 검증
overlap = G_SIGNAL_PROTECT & STOPWORDS_EXTENDED
if overlap:
    print(f"⚠️ G_SIGNAL이 STOPWORDS에 포함됨: {overlap} — 제거 필요")
else:
    print(f"✓ G_SIGNAL 보호 확인 (불용어와 겹침 없음)")

## 3. 토큰화 함수 — exp_F 설정 기준

In [ ]:
def tokenize_kiwi_v2(
    text: str,
    kiwi_instance,
    stopwords: set = None,
    extra_nouns: set = None,
    number_mode: str = 'keep_quantity',
    min_len: int = 2,
) -> list:
    """
    exp_F 설정 기준 토큰화.
    
    설정:
        - stopwords: STOPWORDS_EXTENDED
        - extra_nouns: BOILERPLATE_NOUNS (추가 제거)
        - number_mode: keep_quantity (정량 표현 보존)
        - G_SIGNAL_PROTECT: 어떤 경우에도 제거 금지
    """
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    
    # 1. 숫자 처리
    import re
    if number_mode == 'keep_quantity':
        text = re.sub(r'\d+(\.\d+)?\s*(억원|만원|원|%|퍼센트|톤|tCO2|GWh|MW|kWh)',
                      r'[NUM]\2', text)
        text = re.sub(r'\d+', ' ', text)
    else:
        text = re.sub(r'\d+', ' ', text)
    
    # 2. 불용어 집합 구성 (G_SIGNAL 보호)
    sw = (stopwords or set()) | (extra_nouns or set())
    sw = sw - G_SIGNAL_PROTECT  # G_SIGNAL은 절대 제거 금지
    
    # 3. Kiwi 토큰화
    try:
        tokens_raw = kiwi_instance.tokenize(text)
        tokens = [
            t.form for t in tokens_raw
            if t.tag.startswith('NN')   # 명사만
            and len(t.form) >= min_len  # 최소 길이
            and t.form not in sw        # 불용어 제거
        ]
    except Exception as e:
        tokens = []
    
    return tokens


# 단위 테스트
if KIWI_AVAILABLE:
    test = '재생에너지 투자를 통해 탄소중립을 달성하고 이사회 독립성을 강화합니다.'
    toks = tokenize_kiwi_v2(test, kiwi, STOPWORDS_EXTENDED, BOILERPLATE_NOUNS)
    print(f"Test: {toks}")
    
    for seed in ['재생에너지','탄소중립','이사회']:
        print(f"  '{seed}' 보존: {'✓' if seed in toks else '✗'}")

## 4. 전체 Corpus 토큰화 — exp_F 설정

In [ ]:
# 입력: firm_year_documents_v2.parquet 우선, 없으면 기존 esg_passages.csv
fyd_path = D_PASSAGES / 'firm_year_documents_v2.parquet'
old_passages_path = D_PASSAGES / 'firm_year_documents.csv'

if fyd_path.exists():
    input_df = pd.read_parquet(fyd_path)
    text_col = 'text'
    print(f"firm_year_documents_v2: {input_df.shape}")
elif old_passages_path.exists():
    input_df = pd.read_csv(old_passages_path,
                           dtype={'stock_code': str, 'corp_code': str, 'rcept_no': str})
    input_df['stock_code'] = input_df['stock_code'].astype(str).str.zfill(6)
    text_col = 'text' if 'text' in input_df.columns else input_df.columns[-1]
    print(f"firm_year_documents (기존): {input_df.shape}")
else:
    print("입력 파일 없음 — 01_section_passage_v2.ipynb 먼저 실행")
    input_df = None

if input_df is not None:
    print(f"text_col: {text_col}")
    print(f"avg text length: {input_df[text_col].str.len().mean():.0f} chars")

In [ ]:
if input_df is not None and KIWI_AVAILABLE:
    t0 = time.time()
    
    tokenized_rows = []
    log_rows = []
    
    for i, (_, row) in enumerate(input_df.iterrows()):
        if i % 50 == 0 and i > 0:
            elapsed = time.time() - t0
            rate = i / elapsed
            eta = (len(input_df) - i) / rate
            print(f"  [{i}/{len(input_df)}] {elapsed:.0f}s elapsed, ETA {eta:.0f}s")
        
        text = str(row.get(text_col, '') or '')
        
        # exp_F 설정으로 토큰화
        tokens = tokenize_kiwi_v2(
            text, kiwi,
            stopwords=STOPWORDS_EXTENDED,
            extra_nouns=BOILERPLATE_NOUNS,
            number_mode='keep_quantity',
        )
        
        joined = ' '.join(tokens)
        
        # lineage 보존
        tok_row = {}
        for col in ID_COLS:
            if col in row.index:
                tok_row[col] = row[col]
        tok_row['tokens']       = tokens
        tok_row['joined_text']  = joined
        tok_row['token_count']  = len(tokens)
        tok_row['exp_id']       = 'v_final'
        tok_row['tokenizer']    = f'kiwi_v{__import__("kiwipiepy").__version__ if KIWI_AVAILABLE else "?"}'
        tokenized_rows.append(tok_row)
        
        # seed 보존 체크
        seed_found = [s for s in SEED_30 if s in tokens]
        log_rows.append({
            'stock_code':   row.get('stock_code',''),
            'fiscal_year':  row.get('fiscal_year',''),
            'n_tokens':     len(tokens),
            'n_unique':     len(set(tokens)),
            'n_seed_found': len(seed_found),
            'seed_found':   '|'.join(seed_found[:5]),
        })
    
    elapsed = time.time() - t0
    print(f"\n토큰화 완료: {len(tokenized_rows)}개 문서 in {elapsed:.1f}s")
    
    tok_df = pd.DataFrame(tokenized_rows)
    log_df = pd.DataFrame(log_rows)
    
    # 저장
    tok_path = D_PREPROC / 'tokenized_v_final.parquet'
    tok_df.drop(columns=['tokens'], errors='ignore').to_parquet(tok_path, index=False)
    print(f"→ saved: {tok_path}")
    
    log_path = D_META / 'tokenize_log.csv'
    log_df.to_csv(log_path, index=False, encoding='utf-8-sig')
    print(f"→ saved: {log_path}")
    
    print(f"\n avg token_count: {log_df['n_tokens'].mean():.0f}")
    print(f" avg seed_found:  {log_df['n_seed_found'].mean():.1f}")
elif not KIWI_AVAILABLE:
    print("Kiwi 없음 — 기존 tokenized_exp_F.csv 활용")
    print(f"경로: {D_PREPROC / 'tokenized_exp_F.csv'}")

## 5. Seed 30 보존율 검증 — 핵심 품질 지표

In [ ]:
# tokenized 결과에서 seed 30개 각각의 등장 빈도 확인
# (기존 tokenized_exp_F.csv 사용 가능)
for tok_path_try in [
    D_PREPROC / 'tokenized_v_final.parquet',
    D_PREPROC / 'tokenized_exp_F.csv',
]:
    if tok_path_try.exists():
        if tok_path_try.suffix == '.parquet':
            tok_df = pd.read_parquet(tok_path_try)
        else:
            tok_df = pd.read_csv(tok_path_try)
        print(f"토큰화 결과 로드: {tok_path_try.name} ({tok_df.shape})")
        break

if 'tok_df' in dir() and 'joined_text' in tok_df.columns:
    # 전체 corpus 텍스트 합산
    all_text = ' '.join(tok_df['joined_text'].fillna(''))
    all_tokens = all_text.split()
    
    seed_report = []
    for dim, seeds in {'E': 'SEED_E', 'S': 'SEED_S', 'G': 'SEED_G'}.items():
        from pipeline_config import SEED_E, SEED_S, SEED_G
        seed_list = {'E': SEED_E, 'S': SEED_S, 'G': SEED_G}[dim]
        for s in seed_list:
            freq = all_tokens.count(s)
            doc_count = tok_df['joined_text'].str.contains(s, na=False).sum()
            seed_report.append({
                'dimension': dim, 'seed': s,
                'corpus_freq': freq, 'doc_count': doc_count,
                'rare': freq < 50,
            })
    
    seed_df = pd.DataFrame(seed_report)
    print(f"\n=== Seed 30 Corpus Coverage ===")
    print(f"사전 등록 → corpus 출현: {(seed_df['corpus_freq']>0).sum()}/30")
    print(f"희소 (< 50회): {seed_df['rare'].sum()}개")
    print()
    display(seed_df.sort_values('corpus_freq').to_string())

## 6. Interpretation

**seed 30 보존율이 100%가 아닌 경우:**
- 특정 seed가 0회 등장 → 해당 seed는 feature 계산에서 기여 없음 (그대로 보고)
- '넷제로', '부패방지' 등 희소 seed는 expanded dictionary 확장의 근거

**token_count 분포가 너무 치우친 경우:**
- right-skewed → log_tokens 변환 정당성 확인

**다음 단계:** `03_feature_build_v2.ipynb`  
- tokenized_v_final.parquet → 4종 feature 계산  
- KCGS merge → 회귀 표본 확정

## Robustness Inset — Preprocessing Sensitivity & Tokenizer Instability

이 노트북의 canonical 선택은 **Kiwi · exp_F**다. 그러나 *그 선택이 결과를 어떻게 흔드는가*는 다음 robustness layer에서 검증된다. 본 inset은 결과 두 가지 산출물을 한 셀에서 한 번에 보여주고, raw 분석 노트북으로의 link만 단다 — 재실행 부담을 최소화하기 위해서다.

### 1. Exp_B / E / F preprocessing 비교 — `data/04_preprocessed/eval_comparison.csv`
각 실험에서 *contamination(재무 boilerplate)*과 *ESG specificity*가 어떻게 trade-off되는지를 정량화한다.

### 2. Tokenizer instability — Kiwi vs Okt
동일 corpus · 동일 stopword에서 tokenizer만 바꾸면 vocabulary와 ESG signal이 어떻게 달라지는지는 `../memberC/01_tokenizer_robustness_kiwi_vs_okt.ipynb`에서 분석된다. 본 노트북에서는 비교 figure 한 장과 link로만 짚는다.

### 3. Top-N 토큰 직접 읽기 (평가표 명시 요구)
숫자 지표 위에 "직접 읽은" 판단을 얹는 단계는 `../memberC/01_token_inspection.ipynb`에서 수행된다. BP(재무 boilerplate)·ESG·G-signal 토큰 분류로 어느 실험이 G 분석에 *불가용*한지를 식별한다.


In [ ]:
# === Robustness Inset · 02_preprocess_kiwi_v2 ===
# canonical 선택(Kiwi · exp_F)이 다른 preprocessing 옵션 대비 어떻게 다른지를 산출물에서 직접 로드한다.
import pandas as pd, os
from pathlib import Path
from IPython.display import display, Image, Markdown

ROOT = Path('..').resolve().parent if Path('..').resolve().name=='final' else Path('.').resolve()
# 경로 호환: notebooks/final/에서 실행되는 경우 프로젝트 루트는 두 단계 위.
PROJ = Path.cwd()
while PROJ.name and not (PROJ/'data').exists() and PROJ.parent != PROJ:
    PROJ = PROJ.parent

# 1) Exp 간 contamination/specificity 비교표
eval_path = PROJ/'data'/'04_preprocessed'/'eval_comparison.csv'
if eval_path.exists():
    eval_cmp = pd.read_csv(eval_path)
    display(Markdown('**Preprocessing 실험 비교 (exp_B baseline ↔ exp_E ↔ exp_F canonical):**'))
    display(eval_cmp)
else:
    display(Markdown(f'⚠ `{eval_path}`가 없습니다. 02_preprocess 산출물을 먼저 생성하세요.'))

# 2) preprocessing sensitivity 종합 figure (이미 생성되어 있음)
fig_path = PROJ/'outputs'/'figures'/'figure2_preprocessing_sensitivity.png'
if fig_path.exists():
    display(Markdown('**Figure 2 · Preprocessing Sensitivity (저장된 figure):**'))
    display(Image(str(fig_path)))
else:
    display(Markdown(f'⚠ `{fig_path}`가 없습니다. memberC/02_preprocessing_eval.ipynb를 실행해 생성하세요.'))


### Interpretation — 우리 팀 framing

위 표/figure가 보여주는 것은 "**더 나은 preprocessing**"이 아니라 "**선택의 결과 가변성**"이다.

- BP rate가 낮아져도 ESG rate가 같이 떨어지면 정보를 *버린* 것이지 *정제*한 것이 아니다.
- exp_B → exp_F로 가면서 G 어휘 보존이 일부 떨어지는 실험이 있다 — 이는 후속 단계에서 G-domain 회귀 결과가 약화되는 원인으로 추적된다.
- tokenizer를 바꾸는 것만으로도 동일한 입력에서 ESG-relevant vocabulary가 달라진다 — `memberC/01_tokenizer_robustness` 참조.

**Why this is a finding, not a failure:** preprocessing choice를 어떻게 잡든 ESG signal이 안정적이라면 우리 분석은 *robust*했을 것이다. 그런데 결과는 *흔들린다*. 그 흔들림 자체가 disclosure language의 measurement fragility를 시사한다 — 즉 "ESG 단어를 많이 썼다"는 사실이 가지는 의미가 *분석자의 preprocessing*에 일부 의존한다는 뜻이다. 이는 verbosity dominance · cheap-talk 가능성과 연결된다.

## Decision Box — Preprocessing 선택

- **Alternative:** exp_B (baseline / 최소 정제), exp_E (중간), exp_F (canonical / 강한 정제)
- **Choice:** **exp_F** — Kiwi 형태소 분석기 + 사용자 사전 + 확장 불용어 + ESG-relevance 보존을 명시적으로 검증
- **Justification:** BP contamination 감소와 ESG specificity 유지의 균형 (위 비교표)
- **Limitation:** G 어휘 일부 손실 가능. tokenizer 교체 시 결과 부분적으로 바뀜. 이 한계 자체를 후속 단계에서 *측정*해 보고서에 기록한다.
